# Pipeline Initialization & Environment Guard

This notebook initializes the public preprocessing workflow for large-scale SLURM accounting data before any heavy processing begins.

It does the following:

1. Verifies the expected conda environment (`stat427`).
2. Detects the project root when the notebook is launched from the repository root or `notebooks/`.
3. Uses a local private merged raw input file stored under `data/`.
4. Defines output paths for derived 2025 job-level tables.
5. Creates required output directories if they do not already exist.
6. Confirms that the merged raw input is present before chunk-based processing begins.


In [ ]:
# load necessary libraries and set up environment
import os
import re
import sys
import gzip
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

# Check environment
print("Python executable:", sys.executable)
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV"))
assert os.environ.get("CONDA_DEFAULT_ENV") == "stat427",     "ERROR: Please run this notebook in the stat427 environment created from environment.yml."

# Determine project root
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "environment.yml").exists() and (PROJECT_ROOT.parent / "environment.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Define paths and constants
DATA_DIR = PROJECT_ROOT / "data"
INPUT_GZ = DATA_DIR / "whole_cluster_usage_merged_raw.txt.gz"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
CHUNK_SIZE = 200_000
BUCKET_COUNT = 256
RANDOM_SEED = 427
NA_TOKENS = {"", "unknown", "none", "n/a"}

RAW_VALID_PATH = OUTPUT_DIR / "master_2025_raw_valid.csv.gz"
SUBMISSION_PATH = OUTPUT_DIR / "master_2025_joblevel_submission.csv.gz"
TERMINAL_PATH = OUTPUT_DIR / "master_2025_joblevel_terminal.csv.gz"
TMP_BUCKET_DIR = OUTPUT_DIR / "_tmp_joblevel_buckets"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_GZ.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_GZ}. Place the merged raw SLURM log in data/."
    )

print(f"INPUT_GZ: {INPUT_GZ}")
print(f"PROJECT_ROOT: {PROJECT_ROOT.resolve()}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")
print(f"CHUNK_SIZE: {CHUNK_SIZE:,}")


# Step 1: Validate file schema (header-only read)

To avoid loading the full ~5GB dataset into memory, we read only the header (nrows=0) to inspect the column structure.

This ensures:
- The file is correctly parsed using '|' as separator.
- The expected number of columns (120) is present.
- No structural changes occurred in the raw merged dataset.

This acts as a lightweight schema validation step before chunk-based processing begins.


In [ ]:
# Read the header of the input file to get column names and verify the expected number of columns. 
header_df = pd.read_csv(
    INPUT_GZ,
    sep='|',
    compression='gzip',
    engine='python',
    nrows=0,
)

header_cols = header_df.columns.tolist()
print(f"Header column count: {len(header_cols)}")
assert len(header_cols) == 120, f"Expected 120 columns, got {len(header_cols)}" # Verify that the number of columns matches the expected count, which helps catch issues with file formatting or reading.

print("First 10 columns:", header_cols[:10])
print("Last 10 columns:", header_cols[-10:])


## Header Check Passed

- Column count: **120 (as expected)**
- Delimiter parsing is correct
- Schema appears structurally consistent

The dataset is ready for chunk-based ingestion.

# Step 2: Build `master_2025_raw_valid.csv.gz` (chunked ingest + minimal cleaning)


This step creates a cleaned, analysis-ready dataset of jobs submitted in 2025 from the merged SLURM log.

What it does
1. Standardizes missing values: Treats "", unknown, none, n/a as NA.
   
2. Converts requested memory to MB: Parses ReqMem (e.g., 2G, 500M) into a numeric column ReqMem_MB.

3. Filters to 2025 submissions: Keeps only jobs where `Submit` starts with "2025".

4. Derives scheduling features
	•	WaitTimeSec = Start - Submit
	•	RunTimeSec  = End - Start

5. Simplifies State to its main label (e.g., "CANCELLED by …" → "CANCELLED")

6. Processes the large gz file in chunks to avoid memory issues and writes a gzipped output: `outputs/master_2025_raw_valid.csv.gz`


In [ ]:
import pandas as pd
import numpy as np

EXTRA_CANDIDATES = ["ReqTRES", "Priority", "Eligible", "TimelimitRaw"]
REQUIRED_FOR_FILTER = ["Submit"]  # needed to restrict to 2025 cheaply

MAX_ROWS = 1_000_000
checked_rows = 0
nonempty_counts = {c: 0 for c in EXTRA_CANDIDATES}

def _is_nonempty(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()
    # treat NA_TOKENS and blanks as empty
    return (~s.isna()) & (s.ne("")) & (~s.str.lower().isin(NA_TOKENS))

reader_probe = pd.read_csv(
    INPUT_GZ,
    sep="|",
    compression="gzip",
    engine="python",
    chunksize=CHUNK_SIZE,
    dtype="string",
    on_bad_lines="skip",
    usecols=lambda c: c in (set(EXTRA_CANDIDATES) | set(REQUIRED_FOR_FILTER)),
)

for chunk in reader_probe:
    if "Submit" not in chunk.columns:
        continue
    chunk = chunk[chunk["Submit"].astype("string").str[:4].eq("2025").fillna(False)]
    if chunk.empty:
        continue

    n = len(chunk)
    take = min(n, MAX_ROWS - checked_rows)
    if take <= 0:
        break

    sub = chunk.iloc[:take]

    for c in EXTRA_CANDIDATES:
        if c in sub.columns:
            nonempty_counts[c] += int(_is_nonempty(sub[c]).sum())
        else:
            nonempty_counts[c] += 0

    checked_rows += take
    if checked_rows >= MAX_ROWS:
        break

print("\n=== Column non-empty probe (sampled rows) ===")
print(f"Sampled 2025 rows checked: {checked_rows:,} (max {MAX_ROWS:,})")
rates = {}
for c in EXTRA_CANDIDATES:
    rate = (nonempty_counts[c] / checked_rows) if checked_rows > 0 else 0.0
    rates[c] = rate
    print(f"{c:12s} non-empty: {nonempty_counts[c]:,}  rate: {rate:.4%}")

THRESH = 0.001  # 0.1%
EXTRA_KEEP = [c for c in EXTRA_CANDIDATES if rates.get(c, 0.0) >= THRESH]

print(f"\nKeeping extra cols (rate >= {THRESH:.2%}): {EXTRA_KEEP}")


In [ ]:
# Define functions for data normalization and parsing, as well as a function to count lines in the input file for estimating bad lines.
NA_TOKENS = {"", "unknown", "none", "n/a"}
REQMEM_PATTERN = re.compile(r"^\s*([0-9]*\.?[0-9]+)\s*([kKmMgGtT])(?:[cCnN])?\s*$")

optional_order = ["ReqTRES", "Priority", "Eligible", "TimelimitRaw"]
EXTRA_KEEP = [c for c in optional_order if c in set(globals().get("EXTRA_KEEP", []))]
print(f"EXTRA_KEEP from probe: {EXTRA_KEEP}")

RAW_OUTPUT_COLUMNS = [
    "JobIDRaw", "JobID", "JobName",
    "User", "Group", "Account",
    "Partition", "QOS",
    "Submit",
]
if "Eligible" in EXTRA_KEEP:
    RAW_OUTPUT_COLUMNS.append("Eligible")
if "Priority" in EXTRA_KEEP:
    RAW_OUTPUT_COLUMNS.append("Priority")

RAW_OUTPUT_COLUMNS += [
    "Start", "End",
    "State", "Reason", "ExitCode",
    "ReqCPUS", "ReqMem", "ReqMem_MB", "ReqNodes",
]
if "ReqTRES" in EXTRA_KEEP:
    RAW_OUTPUT_COLUMNS.append("ReqTRES")

RAW_OUTPUT_COLUMNS += [
    "AllocCPUS", "AllocNodes", "AllocTRES",
    "ElapsedRaw",
]
if "TimelimitRaw" in EXTRA_KEEP:
    RAW_OUTPUT_COLUMNS.append("TimelimitRaw")

RAW_OUTPUT_COLUMNS += [
    "WaitTimeSec", "RunTimeSec",
]

for p in [RAW_VALID_PATH, SUBMISSION_PATH, TERMINAL_PATH]:
    if p.exists():
        p.unlink()
if TMP_BUCKET_DIR.exists():
    shutil.rmtree(TMP_BUCKET_DIR)


def normalize_missing(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()
    return s.mask(s.str.lower().isin(NA_TOKENS))


def parse_reqmem_mb(series: pd.Series) -> pd.Series:
    s = normalize_missing(series)
    extracted = s.str.extract(REQMEM_PATTERN)
    numeric = pd.to_numeric(extracted[0], errors='coerce')
    unit = extracted[1].str.upper()
    factor = unit.map({"K": 1/1024, "M": 1, "G": 1024, "T": 1024*1024})
    return numeric * factor


def count_input_data_lines(path: Path) -> int:
    with gzip.open(path, mode='rt', encoding='utf-8', errors='replace') as f:
        _ = next(f, None)  # header
        return sum(1 for _ in f)

print("Counting input lines (for bad-line estimation)...")
input_data_lines = count_input_data_lines(INPUT_GZ)
print(f"Input data lines (without header): {input_data_lines:,}")

parsed_rows = 0
written_rows = 0
header_written = False

normalize_cols = [
    "JobIDRaw", "JobID", "JobName", "User", "Group", "Account",
    "Partition", "QOS", "Submit", "Start", "End", "State",
    "Reason", "ExitCode", "ReqCPUS", "ReqMem", "ReqNodes",
    "AllocCPUS", "AllocNodes", "AllocTRES", "ElapsedRaw",
]
if "Eligible" in EXTRA_KEEP:
    normalize_cols.append("Eligible")
if "Priority" in EXTRA_KEEP:
    normalize_cols.append("Priority")
if "ReqTRES" in EXTRA_KEEP:
    normalize_cols.append("ReqTRES")
if "TimelimitRaw" in EXTRA_KEEP:
    normalize_cols.append("TimelimitRaw")

reader = pd.read_csv(
    INPUT_GZ,
    sep='|',
    compression='gzip',
    engine='python',
    chunksize=CHUNK_SIZE,
    on_bad_lines='skip',
    dtype='string',
)

for chunk_idx, chunk in enumerate(tqdm(reader, desc="Building raw_valid"), start=1):
    parsed_rows += len(chunk)

    for c in normalize_cols:
        if c in chunk.columns:
            chunk[c] = normalize_missing(chunk[c])

    chunk = chunk[chunk["Submit"].str[:4].eq("2025").fillna(False)].copy()
    if chunk.empty:
        continue

    chunk["State"] = normalize_missing(chunk["State"]).str.split().str[0]
    chunk["ReqMem_MB"] = parse_reqmem_mb(chunk["ReqMem"])

    submit_ts = pd.to_datetime(chunk["Submit"], errors='coerce')
    start_ts = pd.to_datetime(chunk["Start"], errors='coerce')
    end_ts = pd.to_datetime(chunk["End"], errors='coerce')

    chunk["WaitTimeSec"] = (start_ts - submit_ts).dt.total_seconds()
    chunk["RunTimeSec"] = (end_ts - start_ts).dt.total_seconds()

    out_chunk = chunk.reindex(columns=RAW_OUTPUT_COLUMNS)
    out_chunk.to_csv(
        RAW_VALID_PATH,
        mode='a',
        index=False,
        header=not header_written,
        compression='gzip',
    )
    header_written = True
    written_rows += len(out_chunk)

    if chunk_idx % 20 == 0:
        print(f"Chunk {chunk_idx}: parsed_rows={parsed_rows:,}, written_rows={written_rows:,}")

bad_line_estimate = max(input_data_lines - parsed_rows, 0)
filter_drop_estimate = parsed_rows - written_rows

print("\nCell 2 summary")
print(f"parsed_rows (after parser, bad lines skipped): {parsed_rows:,}")
print(f"written_rows (2025 raw_valid): {written_rows:,}")
print(f"estimated_bad_lines_skipped: {bad_line_estimate:,}")
print(f"estimated_rows_dropped_by_2025_filter: {filter_drop_estimate:,}")
print(f"raw_valid output: {RAW_VALID_PATH}")


Result: a consistent 2025-only dataset with unified schema and derived timing features,ready for downstream job-level aggregation and analysis.



## Quality statistics for master_2025_raw_valid

This cell performs data quality checks on the cleaned 2025 dataset.
It reports:
   - Total row count and submit time range
   - Normalized state distribution
   - Missing rates for Start and End
   - Proportion of rows with computable WaitTimeSec and RunTimeSec

 This ensures the 2025 cohort, state labels, and derived time features
 are consistent and usable before proceeding to job-level aggregation.

In [ ]:
state_target = ["COMPLETED", "FAILED", "CANCELLED", "TIMEOUT", "OOM", "PENDING", "OTHER"]
state_counts = {k: 0 for k in state_target}

raw_rows = 0
start_missing = 0
end_missing = 0
wait_calc = 0
runtime_calc = 0
submit_min = None
submit_max = None

reader = pd.read_csv(
    RAW_VALID_PATH,
    compression='gzip',
    chunksize=CHUNK_SIZE,
)

for chunk in tqdm(reader, desc="Quality stats"):
    raw_rows += len(chunk)

    submit_ts = pd.to_datetime(chunk["Submit"], errors='coerce')
    chunk_min = submit_ts.min()
    chunk_max = submit_ts.max()
    if pd.notna(chunk_min):
        submit_min = chunk_min if submit_min is None else min(submit_min, chunk_min)
    if pd.notna(chunk_max):
        submit_max = chunk_max if submit_max is None else max(submit_max, chunk_max)

    state_series = chunk["State"].astype("string").str.strip().str.upper()
    state_series = state_series.mask(state_series == "OUT_OF_MEMORY", "OOM") # Normalize "OUT_OF_MEMORY" to "OOM" to consolidate state categories.
    state_series = state_series.where(state_series.isin(state_target[:-1]), "OTHER") # Map any states not in the target list (except for "OTHER") to "OTHER" to consolidate rare or unexpected state categories.
    vc = state_series.value_counts(dropna=False)
    for k, v in vc.items():
        state_counts[k] = state_counts.get(k, 0) + int(v)

    start_missing += int(chunk["Start"].isna().sum())
    end_missing += int(chunk["End"].isna().sum())
    wait_calc += int(pd.to_numeric(chunk["WaitTimeSec"], errors='coerce').notna().sum())
    runtime_calc += int(pd.to_numeric(chunk["RunTimeSec"], errors='coerce').notna().sum())

print("raw_valid_total_rows:", raw_rows)
print("Submit min:", submit_min)
print("Submit max:", submit_max)
print("\nState distribution:")
for k in state_target:
    print(f"  {k}: {state_counts.get(k, 0):,}")

print("\nMissingness:")
print(f"  Start missing rate: {start_missing/raw_rows:.4%}")
print(f"  End missing rate:   {end_missing/raw_rows:.4%}")

print("\nComputable ratios:")
print(f"  WaitTimeSec computable ratio: {wait_calc/raw_rows:.4%}")
print(f"  RunTimeSec computable ratio:  {runtime_calc/raw_rows:.4%}")


1.	2025 submission cohort is complete and stable.
2.	The cluster exhibits a high completion rate (~77%).
3.	A non-trivial fraction of jobs (~10%) are cancelled before execution.
4.	Over 91% of jobs have valid wait and runtime measurements.

# Step 3: Build job-level submission table (one row per JobIDRaw)

We convert the row-level raw_valid table into a job-level "submission" dataset by:
1) Hash-bucketizing rows by JobIDRaw into 256 temporary files (memory-safe grouping),
2) Sorting within each bucket and selecting the earliest Submit record per JobIDRaw,
3) Writing the deduplicated result to master_2025_joblevel_submission.csv.gz.

In [ ]:
TMP_BUCKET_DIR.mkdir(parents=True, exist_ok=True)
for p in TMP_BUCKET_DIR.glob("bucket_*.csv.gz"):
    p.unlink()

if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()

optional_keep = set(globals().get("EXTRA_KEEP", []))

joblevel_work_cols = [
    "JobIDRaw", "JobID", "JobName",
    "User", "Group", "Account",
    "Partition", "QOS",
    "Submit",
]
if "Eligible" in optional_keep:
    joblevel_work_cols.append("Eligible")
if "Priority" in optional_keep:
    joblevel_work_cols.append("Priority")

joblevel_work_cols += [
    "Start", "End",
    "State", "Reason", "ExitCode",
    "ReqCPUS", "ReqMem_MB", "ReqNodes",
]
if "ReqTRES" in optional_keep:
    joblevel_work_cols.append("ReqTRES")

joblevel_work_cols += [
    "AllocCPUS", "AllocNodes", "AllocTRES",
    "ElapsedRaw",
]
if "TimelimitRaw" in optional_keep:
    joblevel_work_cols.append("TimelimitRaw")

joblevel_work_cols += [
    "WaitTimeSec", "RunTimeSec",
]

submission_cols = [
    "JobIDRaw",
    "User", "Group", "Account",
    "Partition", "QOS",
    "Submit",
    "State",
]
if "Priority" in optional_keep:
    submission_cols.append("Priority")

submission_cols += [
    "ReqCPUS", "ReqMem_MB", "ReqNodes",
]
if "ReqTRES" in optional_keep:
    submission_cols.append("ReqTRES")
if "TimelimitRaw" in optional_keep:
    submission_cols.append("TimelimitRaw")

submission_cols += [
    "AllocCPUS", "AllocNodes",
]

row_cursor = 0
bucket_rows = 0

reader = pd.read_csv(
    RAW_VALID_PATH,
    compression='gzip',
    chunksize=CHUNK_SIZE,
    dtype='string',
)

for chunk in tqdm(reader, desc="Bucketizing for job-level"):
    chunk = chunk.reindex(columns=joblevel_work_cols).copy()
    chunk = chunk[chunk["JobIDRaw"].notna()].copy()
    chunk["JobIDRaw"] = chunk["JobIDRaw"].astype("string").str.strip()
    chunk = chunk[chunk["JobIDRaw"].ne("")].copy()

    if chunk.empty:
        continue

    n = len(chunk)
    chunk["_row_order"] = np.arange(row_cursor, row_cursor + n, dtype=np.int64)
    row_cursor += n

    bucket_ids = (
        pd.util.hash_pandas_object(chunk["JobIDRaw"], index=False).astype("uint64") % BUCKET_COUNT
    ).astype("int64")
    chunk["_bucket"] = bucket_ids.values

    for bucket_id, sub in chunk.groupby("_bucket", sort=False):
        path = TMP_BUCKET_DIR / f"bucket_{int(bucket_id):03d}.csv.gz"
        sub.drop(columns=["_bucket"]).to_csv(
            path,
            mode='a',
            index=False,
            header=not path.exists(),
            compression='gzip',
        )
        bucket_rows += len(sub)

print(f"Bucketized rows: {bucket_rows:,}")
print(f"Bucket directory: {TMP_BUCKET_DIR}")

header_written = False
submission_jobs = 0
bucket_files = sorted(TMP_BUCKET_DIR.glob("bucket_*.csv.gz"))

for path in tqdm(bucket_files, desc="Dedup submission per bucket"):
    bdf = pd.read_csv(path, compression='gzip', dtype='string')
    if bdf.empty:
        continue

    bdf["_row_order"] = pd.to_numeric(bdf["_row_order"], errors='coerce').fillna(10**18)
    bdf["_SubmitTS"] = pd.to_datetime(bdf["Submit"], errors='coerce')
    bdf["_SubmitTS_sort"] = bdf["_SubmitTS"].fillna(pd.Timestamp.max)

    bdf = bdf.sort_values(
        ["JobIDRaw", "_SubmitTS_sort", "_row_order"],
        ascending=[True, True, True],
        kind="mergesort",
    )

    selected = bdf.drop_duplicates(subset=["JobIDRaw"], keep="first")
    out = selected.reindex(columns=submission_cols)

    out.to_csv(
        SUBMISSION_PATH,
        mode='a',
        index=False,
        header=not header_written,
        compression='gzip',
    )
    header_written = True
    submission_jobs += len(out)

print(f"submission unique jobs: {submission_jobs:,}")
print(f"submission output: {SUBMISSION_PATH}")


After converting the row-level dataset (raw_valid, 6,059,140 rows) into a job-level submission table:

1. The final submission dataset contains 6,025,326 unique jobs.
2. Only 33,814 rows (~0.56%) were identified as duplicate records across the same JobIDRaw.

# Step 4: Build master_2025_joblevel_terminal (hash buckets + terminal rule)

In [ ]:
if TERMINAL_PATH.exists():
    TERMINAL_PATH.unlink()

optional_keep = set(globals().get("EXTRA_KEEP", []))

terminal_cols = [
    "JobIDRaw",
    "User", "Group", "Account",
    "Partition", "QOS",
    "Submit",
]
if "Eligible" in optional_keep:
    terminal_cols.append("Eligible")

terminal_cols += [
    "State",
]
if "Priority" in optional_keep:
    terminal_cols.append("Priority")

terminal_cols += [
    "ReqCPUS", "ReqMem_MB", "ReqNodes",
]
if "ReqTRES" in optional_keep:
    terminal_cols.append("ReqTRES")

terminal_cols += [
    "AllocCPUS", "AllocNodes",
    "Start", "End", "ElapsedRaw",
]
if "TimelimitRaw" in optional_keep:
    terminal_cols.append("TimelimitRaw")

terminal_cols += [
    "ExitCode", "Reason",
    "WaitTimeSec", "RunTimeSec",
]

header_written = False
terminal_jobs = 0
bucket_files = sorted(TMP_BUCKET_DIR.glob("bucket_*.csv.gz"))

for path in tqdm(bucket_files, desc="Dedup terminal per bucket"):
    bdf = pd.read_csv(path, compression='gzip', dtype='string')
    if bdf.empty:
        continue

    bdf["_row_order"] = pd.to_numeric(bdf["_row_order"], errors='coerce').fillna(10**18)

    bdf["_SubmitTS"] = pd.to_datetime(bdf["Submit"], errors='coerce')
    bdf["_StartTS"] = pd.to_datetime(bdf["Start"], errors='coerce')
    bdf["_EndTS"] = pd.to_datetime(bdf["End"], errors='coerce')

    # priority: 2 (End exists), 1 (Start exists only), 0 (Submit only)
    bdf["_priority"] = np.select(
        [bdf["_EndTS"].notna(), bdf["_StartTS"].notna()],
        [2, 1],
        default=0,
    )

    bdf["_TimeKey"] = bdf["_EndTS"].where(
        bdf["_EndTS"].notna(),
        bdf["_StartTS"].where(bdf["_StartTS"].notna(), bdf["_SubmitTS"]),
    )
    bdf["_TimeKey_sort"] = bdf["_TimeKey"].fillna(pd.Timestamp("1900-01-01"))

    bdf = bdf.sort_values(
        ["JobIDRaw", "_priority", "_TimeKey_sort", "_row_order"],
        ascending=[True, False, False, True],
        kind="mergesort",
    )

    selected = bdf.drop_duplicates(subset=["JobIDRaw"], keep="first")
    out = selected.reindex(columns=terminal_cols)

    out.to_csv(
        TERMINAL_PATH,
        mode='a',
        index=False,
        header=not header_written,
        compression='gzip',
    )
    header_written = True
    terminal_jobs += len(out)

print(f"terminal unique jobs: {terminal_jobs:,}")
print(f"terminal output: {TERMINAL_PATH}")


The submission and terminal tables have identical job counts, indicating no jobs were dropped during aggregation. Data completeness is strong for scheduling analysis: ~91.5% of jobs have computable wait and run times, and end times are missing for only ~0.1% of records.

## Final sanity checks

In [ ]:
ID_TMP_DIR = OUTPUT_DIR / "_tmp_id_compare"
if ID_TMP_DIR.exists():
    shutil.rmtree(ID_TMP_DIR)
ID_TMP_DIR.mkdir(parents=True, exist_ok=True)


def bucketize_jobids(path: Path, prefix: str, chunksize: int = CHUNK_SIZE, bucket_count: int = BUCKET_COUNT):
    for chunk in pd.read_csv(path, compression='gzip', usecols=["JobIDRaw"], chunksize=chunksize, dtype='string'):
        chunk = chunk[chunk["JobIDRaw"].notna()].copy()
        chunk["JobIDRaw"] = chunk["JobIDRaw"].astype("string").str.strip()
        chunk = chunk[chunk["JobIDRaw"].ne("")]
        if chunk.empty:
            continue

        bucket_ids = (
            pd.util.hash_pandas_object(chunk["JobIDRaw"], index=False).astype("uint64") % bucket_count
        ).astype("int64")
        chunk["_bucket"] = bucket_ids.values

        for bid, sub in chunk.groupby("_bucket", sort=False):
            out_path = ID_TMP_DIR / f"{prefix}_{int(bid):03d}.csv.gz"
            sub[["JobIDRaw"]].to_csv(
                out_path,
                mode='a',
                index=False,
                header=not out_path.exists(),
                compression='gzip',
            )


def read_id_set(path: Path) -> set:
    if not path.exists():
        return set()
    df = pd.read_csv(path, compression='gzip', dtype='string')
    if df.empty:
        return set()
    return set(df["JobIDRaw"].dropna().astype(str).str.strip().tolist())


def reservoir_sample_jobids(path: Path, k: int = 5, seed: int = RANDOM_SEED):
    rng = random.Random(seed)
    sample = []
    seen = 0
    for chunk in pd.read_csv(path, compression='gzip', usecols=["JobIDRaw"], chunksize=CHUNK_SIZE, dtype='string'):
        for jid in chunk["JobIDRaw"].dropna().astype(str):
            jid = jid.strip()
            if not jid:
                continue
            seen += 1
            if len(sample) < k:
                sample.append(jid)
            else:
                j = rng.randint(1, seen)
                if j <= k:
                    sample[j - 1] = jid
    return sample


def fetch_rows_for_job(path: Path, job_id: str, columns: list, max_rows: int = 50):
    chunks = []
    for chunk in pd.read_csv(path, compression='gzip', usecols=columns, chunksize=CHUNK_SIZE):
        sub = chunk[chunk["JobIDRaw"].astype(str) == job_id]
        if not sub.empty:
            chunks.append(sub)
    if not chunks:
        return pd.DataFrame(columns=columns)
    out = pd.concat(chunks, ignore_index=True)
    return out.head(max_rows)


print("Bucketing JobIDRaw for set-difference sanity check...")
bucketize_jobids(SUBMISSION_PATH, "sub")
bucketize_jobids(TERMINAL_PATH, "ter")

submission_jobs = 0
terminal_jobs = 0
submission_dup_within_output = 0
terminal_dup_within_output = 0
submission_not_terminal = 0
terminal_not_submission = 0

for i in range(BUCKET_COUNT):
    sub_ids = read_id_set(ID_TMP_DIR / f"sub_{i:03d}.csv.gz")
    ter_ids = read_id_set(ID_TMP_DIR / f"ter_{i:03d}.csv.gz")

    submission_jobs += len(sub_ids)
    terminal_jobs += len(ter_ids)

    submission_not_terminal += len(sub_ids - ter_ids)
    terminal_not_submission += len(ter_ids - sub_ids)

print("\nFinal counts")
print(f"submission job count: {submission_jobs:,}")
print(f"terminal job count:   {terminal_jobs:,}")
print(f"submission_only JobIDRaw count: {submission_not_terminal:,}")
print(f"terminal_only JobIDRaw count:   {terminal_not_submission:,}")

sample_jobids = reservoir_sample_jobids(SUBMISSION_PATH, k=5, seed=RANDOM_SEED)
print("\nSample JobIDRaw for manual audit:", sample_jobids)

optional_keep = set(globals().get("EXTRA_KEEP", []))

RAW_SNAPSHOT_COLS = [
    "JobIDRaw", "JobID", "JobName", "Submit",
]
if "Eligible" in optional_keep:
    RAW_SNAPSHOT_COLS.append("Eligible")
if "Priority" in optional_keep:
    RAW_SNAPSHOT_COLS.append("Priority")

RAW_SNAPSHOT_COLS += [
    "Start", "End", "State", "Reason", "ExitCode",
    "ReqCPUS", "ReqMem_MB", "ReqNodes",
]
if "ReqTRES" in optional_keep:
    RAW_SNAPSHOT_COLS.append("ReqTRES")

RAW_SNAPSHOT_COLS += [
    "AllocCPUS", "AllocNodes", "ElapsedRaw",
]
if "TimelimitRaw" in optional_keep:
    RAW_SNAPSHOT_COLS.append("TimelimitRaw")

RAW_SNAPSHOT_COLS += [
    "WaitTimeSec", "RunTimeSec",
]

SUBMISSION_COLS = [
    "JobIDRaw", "User", "Group", "Account", "Partition", "QOS",
    "Submit", "State",
]
if "Priority" in optional_keep:
    SUBMISSION_COLS.append("Priority")

SUBMISSION_COLS += ["ReqCPUS", "ReqMem_MB", "ReqNodes"]
if "ReqTRES" in optional_keep:
    SUBMISSION_COLS.append("ReqTRES")
if "TimelimitRaw" in optional_keep:
    SUBMISSION_COLS.append("TimelimitRaw")

SUBMISSION_COLS += ["AllocCPUS", "AllocNodes"]

TERMINAL_COLS = [
    "JobIDRaw", "User", "Group", "Account", "Partition", "QOS",
    "Submit",
]
if "Eligible" in optional_keep:
    TERMINAL_COLS.append("Eligible")

TERMINAL_COLS += ["State"]
if "Priority" in optional_keep:
    TERMINAL_COLS.append("Priority")

TERMINAL_COLS += ["ReqCPUS", "ReqMem_MB", "ReqNodes"]
if "ReqTRES" in optional_keep:
    TERMINAL_COLS.append("ReqTRES")

TERMINAL_COLS += [
    "AllocCPUS", "AllocNodes", "Start", "End", "ElapsedRaw",
]
if "TimelimitRaw" in optional_keep:
    TERMINAL_COLS.append("TimelimitRaw")

TERMINAL_COLS += ["ExitCode", "Reason", "WaitTimeSec", "RunTimeSec"]

for jid in sample_jobids:
    print("\n" + "=" * 100)
    print(f"JobIDRaw = {jid}")

    raw_rows = fetch_rows_for_job(RAW_VALID_PATH, jid, RAW_SNAPSHOT_COLS, max_rows=20)
    sub_row = fetch_rows_for_job(SUBMISSION_PATH, jid, SUBMISSION_COLS, max_rows=5)
    ter_row = fetch_rows_for_job(TERMINAL_PATH, jid, TERMINAL_COLS, max_rows=5)

    print("raw_valid snapshots (up to 20 rows):")
    display(raw_rows)

    print("selected submission row:")
    display(sub_row)

    print("selected terminal row:")
    display(ter_row)


In [ ]:
# Optional-column final sanity check
optional_cols = ["ReqTRES", "Priority", "Eligible", "TimelimitRaw"]

raw_cols = pd.read_csv(RAW_VALID_PATH, compression="gzip", nrows=0).columns.tolist()
sub_cols = pd.read_csv(SUBMISSION_PATH, compression="gzip", nrows=0).columns.tolist()
ter_cols = pd.read_csv(TERMINAL_PATH, compression="gzip", nrows=0).columns.tolist()

print("Optional column presence (raw_valid, submission, terminal):")
for c in optional_cols:
    print(c, c in raw_cols, c in sub_cols, c in ter_cols)

if "ReqTRES" in ter_cols:
    gpu_counts = {True: 0, False: 0}
    for chunk in pd.read_csv(
        TERMINAL_PATH,
        compression="gzip",
        usecols=["ReqTRES"],
        chunksize=CHUNK_SIZE,
        dtype="string",
    ):
        contains_gpu = chunk["ReqTRES"].astype("string").str.contains("gpu", case=False, na=False)
        vc = contains_gpu.value_counts()
        gpu_counts[True] += int(vc.get(True, 0))
        gpu_counts[False] += int(vc.get(False, 0))

    print("\nReqTRES contains 'gpu' counts:")
    print(pd.Series(gpu_counts).rename_axis("contains_gpu").sort_index())
else:
    print("\nReqTRES not kept; skipping gpu contains check.")




## Dataset Schema Overview

master_2025_raw_valid.csv.gz

Number of columns: 28

Columns:
	•	JobIDRaw
	•	JobID
	•	JobName
	•	User
	•	Group
	•	Account
	•	Partition
	•	QOS
	•	Submit
	•	Eligible
	•	Priority
	•	Start
	•	End
	•	State
	•	Reason
	•	ExitCode
	•	ReqCPUS
	•	ReqMem
	•	ReqMem_MB
	•	ReqNodes
	•	ReqTRES
	•	AllocCPUS
	•	AllocNodes
	•	AllocTRES
	•	ElapsedRaw
	•	TimelimitRaw
	•	WaitTimeSec
	•	RunTimeSec

Description:
Row-level cleaned snapshot table for jobs submitted in 2025. Includes normalized fields and derived time variables.

⸻

master_2025_joblevel_submission.csv.gz

Number of columns: 16

Columns:
	•	JobIDRaw
	•	User
	•	Group
	•	Account
	•	Partition
	•	QOS
	•	Submit
	•	State
	•	Priority
	•	ReqCPUS
	•	ReqMem_MB
	•	ReqNodes
	•	ReqTRES
	•	TimelimitRaw
	•	AllocCPUS
	•	AllocNodes

Description:
One row per JobIDRaw. Represents the earliest submission snapshot. Used for analyzing submission behavior and requested resources.

⸻

master_2025_joblevel_terminal.csv.gz

Number of columns: 24

Columns:
	•	JobIDRaw
	•	User
	•	Group
	•	Account
	•	Partition
	•	QOS
	•	Submit
	•	Eligible
	•	State
	•	Priority
	•	ReqCPUS
	•	ReqMem_MB
	•	ReqNodes
	•	ReqTRES
	•	AllocCPUS
	•	AllocNodes
	•	Start
	•	End
	•	ElapsedRaw
	•	TimelimitRaw
	•	ExitCode
	•	Reason
	•	WaitTimeSec
	•	RunTimeSec

Description:
One row per JobIDRaw. Represents the final job snapshot and supports execution and scheduling analysis.

⸻
